# EEVR Feature Analysis

Exploratory analysis of the preprocessed, aligned EEVR data (output of `data_preprocessing.ipynb`):

1. Correlation analysis of the EDA and PPG features
2. 2-D feature visualization (t-SNE, UMAP, PaCMAP) coloured by each target label
3. Text analysis (word cloud)
4. Target label distribution

## 1. Environment setup

In [ ]:
import os
os.chdir("/notebooks")

In [ ]:
from lib.install import prepare_env
_ = prepare_env()

In [ ]:
%pip install -q pacmap                                              # not part of lib/install.py

## 2. Imports

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from umap import UMAP
import pacmap
from wordcloud import WordCloud

## 3. Configuration

In [ ]:
DATA_CONFIG = {
    "eda_path":  "EEVR/Data_files/EDA_preprocessed_aligned.csv",
    "ppg_path":  "EEVR/Data_files/PPG_preprocessed_aligned.csv",
    "text_path": "EEVR/Data_files/TEXT_preprocessed_aligned.csv",
}

RANDOM_STATE = 42

label_cols   = ["arousal_category", "valence_category", "taskwiselabel"]
label_titles = ["Arousal", "Valence", "Stimulus-label (CMA)"]

eda_features = [
    "ku_eda", "sk_eda", "dynrange", "slope", "variance",
    "entropy", "insc", "fd_mean", "max_scr", "min_scr",
    "nSCR", "meanAmpSCR", "meanRespSCR", "sumAmpSCR", "sumRespSCR",
]

ppg_features = [
    "BPM", "IBI", "PPG_Rate_Mean",
    "HRV_MedianNN", "HRV_Prc20NN", "HRV_MinNN", "HRV_HTI", "HRV_TINN",
    "HRV_LF", "HRV_VHF", "HRV_LFn", "HRV_HFn", "HRV_LnHF",
    "HRV_SD1SD2", "HRV_CVI", "HRV_PSS", "HRV_PAS", "HRV_PI",
    "HRV_C1d", "HRV_C1a",
    "HRV_DFA_alpha1", "HRV_MFDFA_alpha1_Width", "HRV_MFDFA_alpha1_Peak",
    "HRV_MFDFA_alpha1_Mean", "HRV_MFDFA_alpha1_Max",
    "HRV_MFDFA_alpha1_Delta", "HRV_MFDFA_alpha1_Asymmetry",
    "HRV_ApEn", "HRV_ShanEn", "HRV_FuzzyEn", "HRV_MSEn",
    "HRV_CMSEn", "HRV_RCMSEn",
    "HRV_CD", "HRV_HFD", "HRV_KFD", "HRV_LZC",
]

# ============================================================
# Feature groups, used as boxes on the correlation heatmaps
# start / size are positions in the feature list above
# ============================================================
eda_groups = [
    {"name": "Statistical / signal-level",   "start": 0, "size": 6, "color": "black"},   # E1-E6
    {"name": "Complexity \n / EDA activity", "start": 6, "size": 2, "color": "green"},   # E7-E8
    {"name": "SCR features",                 "start": 8, "size": 7, "color": "purple"},  # E9-E15
]

ppg_groups = [
    {"name": "Heart rate features",           "start": 0,  "size": 3, "color": "black"},       # P1-P3
    {"name": "Time-domain and geometric HRV", "start": 3,  "size": 5, "color": "green"},       # P4-P8
    {"name": "Frequency-domain HRV",          "start": 8,  "size": 5, "color": "purple"},      # P9-P13
    {"name": "Poincare plot features",        "start": 13, "size": 5, "color": "darkorange"},  # P14-P18
    {"name": "Heart rate fragmentation",      "start": 18, "size": 2, "color": "brown"},       # P19-P20
    {"name": "Fractal and multifractal HRV",  "start": 20, "size": 7, "color": "navy"},        # P21-P27
    {"name": "Entropy-based HRV",             "start": 27, "size": 6, "color": "darkred"},     # P28-P33
    {"name": "Nonlinear complexity",          "start": 33, "size": 4, "color": "teal"},        # P34-P37
]

## 4. Read data

In [ ]:
eda_df  = pd.read_csv(DATA_CONFIG["eda_path"])
ppg_df  = pd.read_csv(DATA_CONFIG["ppg_path"])
text_df = pd.read_csv(DATA_CONFIG["text_path"])

print("EDA: ", eda_df.shape)
print("PPG: ", ppg_df.shape)
print("TEXT:", text_df.shape)

## 5. Correlation analysis

Pearson correlation between features, binned into strength classes (|r| < 0.3 is shown as near zero).
Feature families are boxed on the diagonal.

In [ ]:
# ============================================================
# Discrete correlation colour scale
# ============================================================
CORR_THRESHOLD = 0.30

corr_bounds = [-1.0, -0.70, -0.50, -CORR_THRESHOLD, CORR_THRESHOLD, 0.50, 0.70, 1.0]

corr_colors = [
    "#2166ac",    # strong negative
    "#67a9cf",    # moderate negative
    "#d1e5f0",    # weak negative
    "lightgray",  # near zero
    "#fddbc7",    # weak positive
    "#ef8a62",    # moderate positive
    "#b2182b",    # strong positive
]

corr_labels = [
    "Strong negative",
    "Moderate negative",
    "Weak negative",
    "Near zero",
    "Weak positive",
    "Moderate positive",
    "Strong positive",
]


# ============================================================
# Utility: correlation heatmap with feature-group boxes
# ============================================================
def plot_corr_heatmap(df, features, id_prefix, groups, figsize, tick_fontsize,
                      cell_linewidth, box_linewidth, group_fontsize, group_offset, top_margin):
    corr = df[features].corr()

    feature_labels = [
        f"{feature} ({id_prefix}{i})"
        for i, feature in enumerate(features, start=1)
    ]                                                                # ku_eda -> "ku_eda (E1)"

    cmap = ListedColormap(corr_colors)
    norm = BoundaryNorm(corr_bounds, cmap.N)
    tick_positions = [
        (corr_bounds[i] + corr_bounds[i + 1]) / 2
        for i in range(len(corr_bounds) - 1)
    ]                                                                # colorbar tick in each bin centre

    plt.figure(figsize=figsize)

    ax = sns.heatmap(
        corr,
        cmap=cmap,
        norm=norm,
        square=True,
        linewidths=cell_linewidth,
        linecolor="white",
        xticklabels=feature_labels,
        yticklabels=feature_labels,
        cbar_kws={
            "boundaries": corr_bounds,
            "ticks": tick_positions,
            "spacing": "proportional",
        },
    )

    cbar = ax.collections[0].colorbar
    cbar.set_ticklabels(corr_labels)
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("Correlation strength", fontsize=10, fontweight="bold")

    for group in groups:
        start, size, color = group["start"], group["size"], group["color"]

        ax.add_patch(Rectangle(
            (start, start),
            size,
            size,
            fill=False,
            edgecolor=color,
            linewidth=box_linewidth,
        ))                                                           # group box on the diagonal

        ax.text(
            start + size / 2,
            start - group_offset,
            group["name"],
            ha="center",
            va="center",
            fontsize=group_fontsize,
            fontweight="bold",
            color=color,
            bbox=dict(facecolor="white", edgecolor=color, boxstyle="round,pad=0.25", linewidth=1),
            clip_on=False,
        )                                                            # group title above the box

    ax.set_ylim(len(features), -top_margin)                          # room for the top group title
    plt.xticks(rotation=90, ha="right", fontsize=tick_fontsize)
    plt.yticks(rotation=0, fontsize=tick_fontsize)
    plt.tight_layout()
    plt.show()

    return corr

### 5.1 EDA

In [ ]:
eda_corr = plot_corr_heatmap(
    eda_df, eda_features, "E", eda_groups,
    figsize=(12, 10), tick_fontsize=10,
    cell_linewidth=0.5, box_linewidth=1,
    group_fontsize=9, group_offset=0.35, top_margin=1.0,
)

### 5.2 PPG

In [ ]:
ppg_corr = plot_corr_heatmap(
    ppg_df, ppg_features, "P", ppg_groups,
    figsize=(19, 16), tick_fontsize=8,
    cell_linewidth=0.4, box_linewidth=2.5,
    group_fontsize=10, group_offset=0.55, top_margin=2.0,
)

## 6. Feature visualization

Each modality is embedded in 2-D with t-SNE, UMAP and PaCMAP and coloured by each target label.
If a label were easy to predict from the features, its classes would form separate regions.

In [ ]:
# ============================================================
# Utility: one row of scatter plots, one per target label
# ============================================================
def plot_embedding(X_2d, df, method, modality):
    emb_df = pd.DataFrame({"dim1": X_2d[:, 0], "dim2": X_2d[:, 1]})

    for col in label_cols:
        emb_df[col] = df[col].astype(str).values

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, col, title in zip(axes, label_cols, label_titles):
        sns.scatterplot(data=emb_df, x="dim1", y="dim2", hue=col, palette="Set2", ax=ax, s=45)

        ax.set_title(f"{modality} {method}: {title}", fontsize=14, fontweight="bold")
        ax.set_xlabel(f"{method} 1", fontsize=12)
        ax.set_ylabel(f"{method} 2", fontsize=12)
        ax.legend(title=title, fontsize=8, title_fontsize=9)

    plt.tight_layout()
    plt.show()


# ============================================================
# Embedding methods
# ============================================================
def run_tsne(X, **kwargs):
    return TSNE(
        n_components=2,
        perplexity=30,
        learning_rate="auto",
        init="pca",
        random_state=RANDOM_STATE,
        **kwargs,
    ).fit_transform(X)


def run_umap(X):
    return UMAP(
        n_components=2,
        n_neighbors=30,        # larger value gives more global structure
        min_dist=0.00,         # smaller value makes clusters tighter
        metric="cosine",
        spread=1.0,
        random_state=RANDOM_STATE,
    ).fit_transform(X)


def run_pacmap(X):
    X_scaled = StandardScaler().fit_transform(X)                     # visualization only, PaCMAP prefers scaled input

    return pacmap.PaCMAP(
        n_components=2,
        n_neighbors=30,
        MN_ratio=0.5,          # mid-near pairs
        FP_ratio=2.0,          # further pairs
        distance="euclidean",
        random_state=RANDOM_STATE,
    ).fit_transform(X_scaled, init="pca")

### 6.1 EDA

In [ ]:
X_eda = eda_df[eda_features].to_numpy()

plot_embedding(run_tsne(X_eda, max_iter=5000, n_iter_without_progress=500), eda_df, "t-SNE", "EDA")
plot_embedding(run_umap(X_eda), eda_df, "UMAP", "EDA")
plot_embedding(run_pacmap(X_eda), eda_df, "PaCMAP", "EDA")

### 6.2 PPG

In [ ]:
X_ppg = ppg_df[ppg_features].to_numpy()

plot_embedding(run_tsne(X_ppg), ppg_df, "t-SNE", "PPG")
plot_embedding(run_umap(X_ppg), ppg_df, "UMAP", "PPG")
plot_embedding(run_pacmap(X_ppg), ppg_df, "PaCMAP", "PPG")

## 7. Text analysis

Word cloud of the cleaned text (`Text` column: lowercased, stop words removed, lemmatized).
Word size is proportional to how often the word occurs.

In [ ]:
word_freq = Counter(
    word
    for text in text_df["Text"].fillna("").astype(str)
    for word in text.split()
)

print("Unique words:", len(word_freq))
print("Most common:", word_freq.most_common(15))

wordcloud = WordCloud(
    background_color="black",
    width=512,
    height=384,
).generate_from_frequencies(word_freq)

plt.figure(figsize=(15, 15))
plt.imshow(wordcloud)
plt.axis("off")
plt.show()

## 8. Target label analysis

Class counts per target. The imbalance here is why the training notebooks balance the training fold.

In [ ]:
label_colors = ["skyblue", "lightgreen", "salmon"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title, color in zip(axes, label_cols, label_titles, label_colors):
    counts = eda_df[col].value_counts().sort_index()

    bars = ax.bar(counts.index.astype(str), counts.values, color=color, edgecolor="black")
    ax.bar_label(bars, fontsize=13, fontweight="bold", padding=3)    # count above each bar

    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel("Class", fontsize=14)
    ax.set_ylabel("Count", fontsize=14)
    ax.tick_params(axis="both", labelsize=13)
    ax.margins(y=0.12)                                               # headroom for the count labels

plt.tight_layout()
plt.show()